# Retail Site Selection - Sales Prediction Model

Predicts annual sales for expansion candidates using XGBoost trained on existing store data.

**Approach:**
- Train XGBoost on all ~546 stores across 7 states (MI, VA, NY, WA, MD, NJ, MA)
- Evaluate with stratified 5-fold cross-validation (stratified by state, ~109 stores per fold)
- Log-transform target variable to handle sales variance
- Use SHAP for model explainability
- Register production model to Unity Catalog with "Champion" alias

**Why train on all data?** Stratified CV provides honest, held-out evaluation metrics.
The final model then trains on ALL stores to maximize prediction quality for candidates,
which are truly unseen locations with no sales data.

**Inputs:**
- `current_stores_features_agg` - Existing stores with aggregated trade area features and sales
- `candidates_features_agg` - Candidate locations with aggregated trade area features

**Outputs:**
- `candidates_finalized` - Ranked candidates with predicted sales
- `sales_prediction_model` - Registered MLflow model (Unity Catalog)

## Setup

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

# Parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("gold_schema", "")
dbutils.widgets.text("min_predicted_sales", "250000")
dbutils.widgets.text("min_population", "5000")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
min_predicted_sales = int(dbutils.widgets.get("min_predicted_sales"))
min_population = int(dbutils.widgets.get("min_population"))

# Table names
training_table = f"{catalog}.{gold_schema}.current_stores_features_agg"
candidates_table = f"{catalog}.{gold_schema}.candidates_features_agg"
output_table = f"{catalog}.{gold_schema}.candidates_finalized"
model_name = f"{catalog}.{gold_schema}.sales_prediction_model"

print(f"Training data: {training_table}")
print(f"Candidates: {candidates_table}")
print(f"Output: {output_table}")
print(f"Min predicted sales: ${min_predicted_sales:,}")
print(f"Min population: {min_population:,}")

# Set MLflow experiment
mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/geospatial-retail-sales-prediction")

# Close any orphaned runs from previous executions
mlflow.end_run()

# Start parent pipeline run (all subsequent runs are nested under this)
parent_run = mlflow.start_run(run_name="sales_prediction_pipeline")
mlflow.set_tags({
    "pipeline": "sales_prediction",
    "layer": "gold",
    "notebook": "predict_candidate_sales",
})
client = MlflowClient()

print(f"\nMLflow experiment configured")
print(f"Parent run: {parent_run.info.run_id}")

## 1. Load and Prepare Data

In [ ]:
# All features - no RFE, XGBoost's L1/L2 regularization handles feature selection
feature_columns = [
    # Demographics
    'population',
    'target_demographic_total',

    # POI counts by category
    'retail', 'food_drink', 'leisure', 'education',
    'healthcare', 'financial', 'tourism', 'transportation',

    # Competition and co-location
    'competitor_count',
    'partner_count',

    # Activity indicator
    'human_activity_index',

    # Trade area size
    'h3_cell_count',
    'area_sqkm',
]

target_column = 'annual_sales'

print(f"Features ({len(feature_columns)}): {feature_columns}")
print(f"Target: {target_column}")

In [ ]:
# Load training data
training_df = spark.table(training_table)
print(f"Loaded {training_df.count()} existing stores for training")

# Log training dataset for lineage tracking
training_dataset = mlflow.data.from_spark(training_df, table_name=training_table, version="0")
mlflow.log_input(training_dataset, context="training")

print("\nStore distribution by state:")
display(training_df.groupBy("state").count().orderBy("state"))

# Convert to Pandas
train_pd = training_df.select(feature_columns + [target_column, 'store_number', 'city', 'state']).toPandas()

# Normalize state values to abbreviations
state_mapping = {
    'Massachusetts': 'MA', 'massachusetts': 'MA',
    'Connecticut': 'CT', 'connecticut': 'CT',
    'New Jersey': 'NJ', 'new jersey': 'NJ',
    'Maryland': 'MD', 'maryland': 'MD',
    'Michigan': 'MI', 'michigan': 'MI',
    'Virginia': 'VA', 'virginia': 'VA',
    'New York': 'NY', 'new york': 'NY',
    'Washington': 'WA', 'washington': 'WA',
    'MA': 'MA', 'CT': 'CT', 'NJ': 'NJ', 'MD': 'MD',
    'MI': 'MI', 'VA': 'VA', 'NY': 'NY', 'WA': 'WA',
}
train_pd['state'] = train_pd['state'].map(lambda x: state_mapping.get(x, x))

print(f"\nState distribution after normalization:")
print(train_pd['state'].value_counts())

train_encoded = train_pd.copy()

print(f"\nData shape: {train_encoded.shape}")
print(f"Sample:")
display(train_encoded.head())

## 2. Correlation Analysis

In [ ]:
# Prepare features (numeric only)
X_initial = train_encoded.drop(columns=[target_column, 'store_number', 'city', 'state'])
X_initial = X_initial.select_dtypes(include=['number'])
y = train_encoded[target_column]

print(f"Initial feature set: {list(X_initial.columns)}")
print(f"Total features: {len(X_initial.columns)}")

# Correlation matrix
correlation_matrix = X_initial.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Find highly correlated pairs (|r| > 0.85)
high_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.85:
            high_corr.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if high_corr:
    print(f"\nHighly correlated pairs (|r| > 0.85): {len(high_corr)}")
    for feat1, feat2, corr in high_corr[:10]:
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")
else:
    print("\n✓ No high multicollinearity detected")

# Correlation with target
target_corr = X_initial.corrwith(y).sort_values(ascending=False)
print(f"\nTop features correlated with {target_column}:")
print(target_corr)

## 3. Feature Set

Using all 15 features directly — XGBoost's built-in L1/L2 regularization handles feature selection
without needing explicit RFE.

In [ ]:
# Use all features directly (no RFE - regularized XGBoost handles feature selection)
final_features = list(feature_columns)
X_final = X_initial[final_features]

print(f"{'='*60}")
print(f"FEATURE SET: {len(final_features)} features")
print(f"{'='*60}")
for i, feat in enumerate(final_features, 1):
    print(f"  {i:2d}. {feat}")

In [ ]:
# Prepare target variable with log transformation
y_all = y.copy()
y_all_log = np.log1p(y_all)
state_labels = train_encoded['state']

print("=" * 60)
print("DATA SUMMARY")
print("=" * 60)
print(f"\nStores: {len(X_final)} across {state_labels.nunique()} states")
print(f"Features: {len(final_features)}")
print(f"\nSales distribution:")
print(f"  Min: ${y_all.min():,.0f}")
print(f"  Max: ${y_all.max():,.0f}")
print(f"  Mean: ${y_all.mean():,.0f}")
print(f"  Std: ${y_all.std():,.0f}")
print(f"  CV: {y_all.std()/y_all.mean():.2f}")
print(f"\nStores by state:")
print(state_labels.value_counts().sort_index())

# Visualize distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y_all, bins=20, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Annual Sales ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sales Distribution (Original)')

axes[1].hist(y_all_log, bins=20, alpha=0.7, color='coral', edgecolor='white')
axes[1].set_xlabel('Log(1 + Annual Sales)')
axes[1].set_ylabel('Count')
axes[1].set_title('Sales Distribution (Log-Transformed)')

plt.tight_layout()
plt.show()

## 4. Cross-Validation

Stratified 5-fold CV evaluates model performance on held-out data. Each fold holds out ~109 stores
(stratified by state so every fold has proportional representation from all 7 states).
These metrics are the honest estimate of how well the model generalizes.

In [ ]:
# Stratified 5-Fold Cross-Validation (XGBoost)
# Each fold holds out ~109 stores, stratified by state for proportional representation

print("=" * 60)
print("STRATIFIED 5-FOLD CROSS-VALIDATION")
print("=" * 60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results_xgb = []
oof_pred_xgb = np.zeros(len(y_all))

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_final, state_labels)):
    X_train_fold = X_final.iloc[train_idx]
    X_val_fold = X_final.iloc[val_idx]
    y_train_fold_log = y_all_log.iloc[train_idx]
    y_val_original = y_all.iloc[val_idx]

    xgb_fold = xgb.XGBRegressor(
        n_estimators=100, max_depth=3, min_child_weight=5,
        learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1
    )
    xgb_fold.fit(X_train_fold, y_train_fold_log)
    pred_xgb = np.maximum(0, np.expm1(xgb_fold.predict(X_val_fold)))
    oof_pred_xgb[val_idx] = pred_xgb
    cv_results_xgb.append({
        'fold': fold_idx + 1,
        'rmse': np.sqrt(mean_squared_error(y_val_original, pred_xgb)),
        'r2': r2_score(y_val_original, pred_xgb),
        'mae': mean_absolute_error(y_val_original, pred_xgb),
        'n_val': len(val_idx)
    })

    print(f"\nFold {fold_idx + 1}: {len(train_idx)} train / {len(val_idx)} val")
    print(f"  RMSE: ${cv_results_xgb[-1]['rmse']:,.0f}, R²: {cv_results_xgb[-1]['r2']:.3f}")

cv_xgb_df = pd.DataFrame(cv_results_xgb)

print(f"\n{'='*60}")
print("5-FOLD CV SUMMARY")
print("=" * 60)
print(f"  Mean RMSE: ${cv_xgb_df['rmse'].mean():,.0f} (+-${cv_xgb_df['rmse'].std():,.0f})")
print(f"  Mean R²:   {cv_xgb_df['r2'].mean():.3f} (+-{cv_xgb_df['r2'].std():.3f})")

# Out-of-fold diagnostic plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_all, oof_pred_xgb, alpha=0.3, s=20)
ax.plot([y_all.min(), y_all.max()], [y_all.min(), y_all.max()], 'r--')
ax.set_xlabel('Actual Sales ($)')
ax.set_ylabel('Predicted Sales ($)')
ax.set_title(f'XGBoost Out-of-Fold Predictions (R²={cv_xgb_df["r2"].mean():.3f})')
plt.tight_layout()
plt.show()

# Log to MLflow (nested under parent pipeline run)
with mlflow.start_run(run_name="stratified_5fold_cv", nested=True):
    mlflow.log_param("cv_strategy", "stratified_5fold")
    mlflow.log_param("n_splits", 5)
    mlflow.log_param("features", ",".join(final_features))
    mlflow.log_param("n_features", len(final_features))
    mlflow.log_param("n_stores", len(X_final))
    mlflow.log_metric("xgb_mean_cv_rmse", cv_xgb_df['rmse'].mean())
    mlflow.log_metric("xgb_mean_cv_r2", cv_xgb_df['r2'].mean())
    mlflow.log_figure(fig, "cv_diagnostic_plot.png")
    mlflow.log_table(cv_xgb_df, artifact_file="cv_results_xgb.json")

display(cv_xgb_df)

## 5. Train Production Model

Cross-validation above evaluated model quality on held-out folds (~109 stores each).
Now we train the final XGBoost on **all** stores to maximize prediction accuracy for candidates.

In [ ]:
with mlflow.start_run(run_name="final_xgboost", nested=True):
    xgb_model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=3,
        min_child_weight=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    )
    xgb_model.fit(X_final, y_all_log)

    y_pred_all_log = xgb_model.predict(X_final)
    y_pred_all = np.expm1(y_pred_all_log)
    train_rmse = np.sqrt(mean_squared_error(y_all, y_pred_all))

    mlflow.log_params({
        "model_type": "XGBRegressor",
        "target_transform": "log1p",
        "n_estimators": 100,
        "max_depth": 3,
        "min_child_weight": 5,
        "learning_rate": 0.1,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "n_features": len(final_features),
        "train_size": len(X_final),
        "cv_strategy": "stratified_5fold"
    })

    mlflow.log_metrics({
        "train_rmse": train_rmse,
        "cv_mean_rmse": cv_xgb_df['rmse'].mean(),
        "cv_mean_r2": cv_xgb_df['r2'].mean(),
        "cv_std_r2": cv_xgb_df['r2'].std()
    })

    # Feature importance
    fig, ax = plt.subplots(figsize=(10, 6))
    xgb.plot_importance(xgb_model, ax=ax, max_num_features=15, importance_type='gain')
    plt.title('XGBoost Feature Importance (Gain)')
    plt.tight_layout()
    mlflow.log_figure(fig, "xgb_feature_importance.png")
    plt.show()

    mlflow.sklearn.log_model(xgb_model, "model")

    xgb_metrics = {
        "cv_rmse": cv_xgb_df['rmse'].mean(),
        "cv_r2": cv_xgb_df['r2'].mean(),
        "cv_r2_std": cv_xgb_df['r2'].std()
    }

    print(f"\nFinal XGBoost (trained on all {len(X_final)} stores):")
    print(f"  Train RMSE: ${train_rmse:,.0f}")
    print(f"  CV Mean RMSE: ${xgb_metrics['cv_rmse']:,.0f}")
    print(f"  CV Mean R²: {xgb_metrics['cv_r2']:.3f} (+-{xgb_metrics['cv_r2_std']:.3f})")

## 6. Explainability (SHAP)

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions.
- **Summary plot**: Which features matter most globally, and their directional effects
- **Bar chart**: Mean absolute SHAP values ranked by importance
- **Waterfall**: Feature contributions for a single store prediction

In [ ]:
# Create SHAP explainer for XGBoost (using all training data)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_final)

print("SHAP values computed for all stores")
print(f"Shape: {shap_values.shape}")

In [ ]:
# SHAP Summary Plot (global feature importance)
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, X_final, show=False)
plt.title('SHAP Summary Plot - Feature Impact on Sales Predictions')
plt.tight_layout()
mlflow.log_figure(fig, "shap_summary_plot.png")
plt.show()

print("\nInterpretation:")
print("- Features listed by importance (top to bottom)")
print("- Red = high feature value, Blue = low feature value")
print("- Position on x-axis shows impact on prediction")

In [ ]:
# SHAP Feature Importance Bar Chart
shap_importance = pd.DataFrame({
    'feature': X_final.columns,
    'mean_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_shap', ascending=False)

print("Top Features by SHAP Importance:")
print(shap_importance)

fig, ax = plt.subplots(figsize=(10, 6))
shap_importance.plot(x='feature', y='mean_shap', kind='barh', ax=ax, color='steelblue')
plt.xlabel('Mean |SHAP value|')
plt.ylabel('Feature')
plt.title('Features by SHAP Importance')
plt.tight_layout()
mlflow.log_figure(fig, "shap_importance_bar.png")
plt.show()

In [ ]:
# SHAP Waterfall Plot (explain single store prediction)
sample_idx = 0
fig, ax = plt.subplots(figsize=(10, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[sample_idx],
        base_values=explainer.expected_value,
        data=X_final.iloc[sample_idx],
        feature_names=X_final.columns.tolist()
    ),
    show=False
)
plt.title(f'SHAP Waterfall: Feature contributions for store #{sample_idx}')
plt.tight_layout()
mlflow.log_figure(fig, "shap_waterfall_sample.png")
plt.show()

print(f"\nActual sales: ${y_all.iloc[sample_idx]:,.0f}")
print(f"Out-of-fold predicted: ${oof_pred_xgb[sample_idx]:,.0f}")

## 7. Register Model

Register the production XGBoost model to Unity Catalog and set the "Champion" alias
for downstream consumption by the application layer.

In [ ]:
# Register production model to Unity Catalog
with mlflow.start_run(run_name="PRODUCTION_XGBoost", nested=True):
    mlflow.set_tags({
        "model_type": "XGBoost",
        "stage": "production",
    })
    mlflow.log_params({
        "model_type": "XGBoost",
        "features": final_features
    })

    model_info = mlflow.sklearn.log_model(
        xgb_model,
        "model",
        registered_model_name=model_name,
        input_example=X_final.iloc[:5],
        signature=mlflow.models.infer_signature(X_final, y_all)
    )

# Set Champion alias on the registered version
client.set_registered_model_alias(model_name, "Champion", model_info.registered_model_version)

print(f"\nModel registered to Unity Catalog:")
print(f"  Name: {model_name}")
print(f"  Type: XGBoost")
print(f"  Version: {model_info.registered_model_version}")
print(f"  Alias: Champion")

## 8. Predict Candidate Sales

In [ ]:
# Load candidates with aggregated features
candidates_df = spark.table(candidates_table)
print(f"Loaded {candidates_df.count():,} expansion candidates")

print("\nCandidates by state:")
display(candidates_df.groupBy("state").count().orderBy("state"))

# Convert to Pandas and prepare features
candidates_pd = candidates_df.select(['candidate_id'] + feature_columns + ['state']).toPandas()

# Normalize state values
state_mapping = {
    'Massachusetts': 'MA', 'massachusetts': 'MA',
    'Connecticut': 'CT', 'connecticut': 'CT',
    'New Jersey': 'NJ', 'new jersey': 'NJ',
    'Maryland': 'MD', 'maryland': 'MD',
    'Michigan': 'MI', 'michigan': 'MI',
    'Virginia': 'VA', 'virginia': 'VA',
    'New York': 'NY', 'new york': 'NY',
    'Washington': 'WA', 'washington': 'WA',
    'MA': 'MA', 'CT': 'CT', 'NJ': 'NJ', 'MD': 'MD',
    'MI': 'MI', 'VA': 'VA', 'NY': 'NY', 'WA': 'WA',
}
candidates_pd['state'] = candidates_pd['state'].map(lambda x: state_mapping.get(x, x))

candidates_encoded = candidates_pd.copy()

# Align columns with training data
missing_cols = set(final_features) - set(candidates_encoded.columns)
for c in missing_cols:
    candidates_encoded[c] = 0
    print(f"Warning: Missing column '{c}' - filled with 0")

X_candidates = candidates_encoded[['candidate_id'] + final_features]

print(f"\nPrepared {len(X_candidates)} candidates for prediction")
print(f"Using features: {final_features}")

In [ ]:
# Predict sales for candidates using production XGBoost model
X_pred = X_candidates.drop(columns=['candidate_id'])

# Predictions in log scale, then convert back
predictions_log = xgb_model.predict(X_pred)
predictions = np.expm1(predictions_log)
predictions = np.maximum(predictions, 0)  # Floor at 0

# Prediction intervals based on CV RMSE
cv_rmse = xgb_metrics['cv_rmse']

results_pd = pd.DataFrame({
    'candidate_id': X_candidates['candidate_id'],
    'predicted_annual_sales': predictions.astype(int),
    'predicted_log_sales': predictions_log,
    'prediction_interval_lower': np.maximum(0, (predictions - cv_rmse)).astype(int),
    'prediction_interval_upper': (predictions + cv_rmse).astype(int),
    'model_version': "XGBoost_log_scaled"
})

print(f"\nPrediction Summary:")
print(f"  Count: {len(results_pd):,}")
print(f"  Mean: ${results_pd['predicted_annual_sales'].mean():,.0f}")
print(f"  Median: ${results_pd['predicted_annual_sales'].median():,.0f}")
print(f"  Min: ${results_pd['predicted_annual_sales'].min():,.0f}")
print(f"  Max: ${results_pd['predicted_annual_sales'].max():,.0f}")

display(results_pd.head(10))

## 9. Rank & Write Output

In [ ]:
# Convert predictions to Spark and join back to candidates
predictions_spark = spark.createDataFrame(results_pd)

candidates_with_predictions = candidates_df.join(
    predictions_spark,
    "candidate_id",
    "inner"
).withColumn(
    "predicted_monthly_sales",
    (col("predicted_annual_sales") / 12).cast("long")
)

print(f"Joined predictions to {candidates_with_predictions.count():,} candidates")

# Apply business constraints (currently disabled)
candidates_constrained = candidates_with_predictions

before_count = candidates_with_predictions.count()
after_count = candidates_constrained.count()
print(f"\nBusiness constraints:")
print(f"  Before: {before_count:,}")
print(f"  After: {after_count:,}")
print(f"  Filtered: {before_count - after_count:,}")

# Rank candidates by predicted sales
window_spec = Window.orderBy(F.desc("predicted_annual_sales"))
candidates_ranked = candidates_constrained.withColumn(
    "rank", F.row_number().over(window_spec)
).withColumn(
    "percentile", F.percent_rank().over(window_spec)
)

# Add metadata
candidates_final = candidates_ranked.withColumn(
    "prediction_timestamp", F.current_timestamp()
).withColumn(
    "min_sales_threshold", F.lit(min_predicted_sales)
).withColumn(
    "min_population_threshold", F.lit(min_population)
)

# Write to gold
(
    candidates_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"\n✓ Written {candidates_final.count():,} ranked candidates to {output_table}")

# Show top candidates
print("\nTop 20 Expansion Candidates:")
display(candidates_final.select(
    "rank", "candidate_id", "latitude", "longitude", "state", "urbanity",
    "population", "target_demographic_total", "total_poi_count",
    "predicted_annual_sales", "predicted_monthly_sales"
).limit(20))

## Summary

In [ ]:
# End parent pipeline run
mlflow.end_run()

print("=" * 60)
print("SALES PREDICTION MODEL - SUMMARY")
print("=" * 60)

print(f"\n1. Training Data:")
print(f"   - {len(train_encoded)} stores across {state_labels.nunique()} states")

print(f"\n2. Features: {len(final_features)}")
for f in final_features:
    print(f"   - {f}")

print(f"\n3. XGBoost 5-Fold CV:")
print(f"   - Mean R²: {cv_xgb_df['r2'].mean():.3f} (+-{cv_xgb_df['r2'].std():.3f})")
print(f"   - Mean RMSE: ${cv_xgb_df['rmse'].mean():,.0f}")

print(f"\n4. Production Model:")
print(f"   - Registered: {model_name}")
print(f"   - Alias: Champion")

print(f"\n5. Predictions:")
print(f"   - {len(results_pd):,} candidates scored")
print(f"   - Output: {output_table}")

print(f"\n{'='*60}")
print("COMPLETE")
print("=" * 60)